In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
!pip install tensorflow opencv-python pandas numpy scikit-learn matplotlib seaborn kaggle

In [3]:
!pip uninstall -y jax jaxlib

Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [5]:
import tensorflow as tf

train_dir = "/content/drive/MyDrive/archive (1)/train"
test_dir = "/content/drive/MyDrive/archive (1)/test"

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=(48,48),
    batch_size=32,
    color_mode="grayscale"
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=(48,48),
    batch_size=32,
    color_mode="grayscale"
)

Found 28709 files belonging to 7 classes.
Found 1135 files belonging to 7 classes.


In [7]:
import os

print("Train classes:", os.listdir("/content/drive/MyDrive/archive (1)/train"))
print("Test classes:", os.listdir("/content/drive/MyDrive/archive (1)/test"))

Train classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Test classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [8]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_data = train_data.map(lambda x, y: (normalization_layer(x), y))
test_data = test_data.map(lambda x, y: (normalization_layer(x), y))

In [9]:
from tensorflow.keras import layers, models

model = models.Sequential([

    layers.Conv2D(32,(3,3),activation='relu',input_shape=(48,48,1)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(128,activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(7,activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 355,847 (1.36 MB)

 Trainable params: 355,847 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
history = model.fit(
    train_data,
    epochs=40,
    validation_data=test_data
)

Epoch 1/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 5498s 6s/step - accuracy: 0.2973 - loss: 1.7323 - val_accuracy: 0.1586 - val_loss: 1.8988
Epoch 2/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.4186 - loss: 1.5064 - val_accuracy: 0.1348 - val_loss: 1.8439
Epoch 3/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 47s 53ms/step - accuracy: 0.4675 - loss: 1.3864 - val_accuracy: 0.0890 - val_loss: 1.9964
Epoch 4/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 48s 53ms/step - accuracy: 0.5008 - loss: 1.3088 - val_accuracy: 0.0890 - val_loss: 1.9996
Epoch 5/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.5227 - loss: 1.2661 - val_accuracy: 0.1639 - val_loss: 1.7900
Epoch 6/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.5401 - loss: 1.2099 - val_accuracy: 0.1762 - val_loss: 1.7844
Epoch 7/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.5549 - loss: 1.1759 - val_accuracy: 0.1885 - val_loss: 1.7400
Epoch 8/40
898/898 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.5678 - loss: 1.1367 - 

In [12]:
model.save("emotion_model.h5")
print("Model saved successfully!")

Model saved successfully!


In [13]:
class_names = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

In [14]:
import cv2
import numpy as np

def predict_emotion(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (48,48))

    img = img / 255.0
    img = img.reshape(1,48,48,1)

    prediction = model.predict(img)
    emotion_index = np.argmax(prediction)

    return class_names[emotion_index]

In [17]:
from google.colab import files
uploaded = files.upload()

Saving img.png to img.png


In [18]:
image_path = list(uploaded.keys())[0]

emotion = predict_emotion(image_path)

print("Detected Emotion:", emotion)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 612ms/step
Detected Emotion: sad


In [19]:
import pandas as pd

music = pd.read_csv('/content/drive/MyDrive/Music dataset.csv/Data/spotify_songs.csv')

music.head()

,name,artists,energy,key,mode,valence,tempo,duration_ms
0,God's Plan,Drake,0.449,7,1,0.357,77.169,198973
1,SAD!,XXXTENTACION,0.613,8,1,0.473,75.023,166606
2,rockstar (feat. 21 Savage),Post Malone,0.535,5,0,0.140,159.847,218147
3,Psycho (feat. Ty Dolla $ign),Post Malone,0.559,8,1,0.439,140.124,221440
4,In My Feelings,Drake,0.626,1,1,0.350,91.030,217925


In [20]:
print(music.columns)

Index(['name', 'artists', 'energy', 'key', 'mode', 'valence', 'tempo',
       'duration_ms'],
      dtype='object')


In [21]:
def recommend_music(emotion):

    emotion_music_map = {
        "angry": "rock",
        "disgust": "pop",
        "fear": "ambient",
        "happy": "pop",
        "neutral": "edm",
        "sad": "acoustic",
        "surprise": "dance"
    }

    mood = emotion_music_map[emotion]

    results = music[music['playlist_genre'].str.contains(mood, case=False, na=False)]

    return results[['track_name','track_artist']].head(5)

In [23]:
def recommend_music(emotion):

    if emotion == "happy":
        results = music[(music['valence'] > 0.6) & (music['energy'] > 0.5)]

    elif emotion == "sad":
        results = music[(music['valence'] < 0.4) & (music['energy'] < 0.5)]

    elif emotion == "angry":
        results = music[(music['energy'] > 0.7)]

    elif emotion == "neutral":
        results = music[(music['valence'].between(0.4,0.6))]

    elif emotion == "fear":
        results = music[(music['energy'] < 0.4)]

    elif emotion == "surprise":
        results = music[(music['energy'] > 0.6) & (music['valence'] > 0.5)]

    elif emotion == "disgust":
        results = music[(music['energy'] < 0.5)]

    return results[['name','artists']].head(5)

In [24]:
songs = recommend_music(emotion)

print("\n Recommended Songs for", emotion, ":\n")
print(songs)


 Recommended Songs for sad :

                                      name        artists
0                               God's Plan          Drake
29                                 Perfect     Ed Sheeran
50         Ric Flair Drip (& Metro Boomin)         Offset
55  FEFE (feat. Nicki Minaj & Murda Beatz)        6ix9ine
80                    lovely (with Khalid)  Billie Eilish
